# AG03 · Tools Done Right

**Track:** Agents (lesson 3 of 15) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** AG02 (the agent loop) · **You will build:** a tool-schema **linter**, five repaired tools, and an experiment measuring tool-selection accuracy

---
## 🎯 Why this matters
In AG02 the model was the star and the tools were two toy functions. In production it is usually the opposite: **the model is a commodity, and the quality of your tools decides whether the agent works.**
A tool is a contract between a deterministic system (your code) and a non-deterministic caller (the model). The model sees *only* the name, the description and the schema: nothing else. If those are vague, it will guess; if the errors are cryptic, it cannot recover; if a retry double-charges a customer, you will find out from finance.

**Real scenario.** *Kauri Outdoors*, an NZ outdoor-gear retailer, has an internal support agent whose five tools were written in an afternoon: `run`, `get_info`, `update`, `search`, `refund_order`. It mis-picks tools, floods its own context with 60 KB search results, and once refunded the same order twice after a network timeout. You will lint, repair and *measure* your way out of that.

## 🧭 Learning objectives
By the end you can:
1. Write tool names, descriptions and schemas (enums, documented params, strict objects) that steer a model toward correct calls.
2. Implement a **tool linter** that scores a schema 0-100 against twelve explicit rules, and unit-test it.
3. Design **recoverable errors**, **idempotent writes with dry-run**, and **paginated, truncation-aware reads**.
4. Measure how description quality and tool count change selection accuracy using an offline policy model, and know what that proxy can and cannot tell you.
5. Detect near-duplicate tools across a whole toolbox.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: the agent-computer interface (ACI)
We spend enormous effort on human-computer interfaces (HCI); an agent's tools are its interface to your systems, and deserve the same care. Anthropic's *Building effective agents* calls this the **agent-computer interface** and recommends investing in it as you would in a UI: put yourself in the model's shoes ("is it obvious how to use this tool from the description and parameters alone, or would I need to think hard about it?"), and make mistakes hard to make (*poka-yoke*). Their example: for a coding agent they changed a tool to require *absolute* file paths, which removed a class of errors once the agent moved out of the root directory.

**Principles used in this lesson**

| # | Principle | Mechanism | Rule in our linter |
|---|---|---|---|
| 1 | Name tells *verb + object* | `get_order_status`, not `get_info` | R1, R2 |
| 2 | Description says **when to use**, **when not to**, and **what comes back** | 3-4 sentences | R3-R6 |
| 3 | Every parameter is documented, unambiguous, and typed tightly | no `data`, `id`, `x` | R7, R8 |
| 4 | Closed sets are **enums**, not free text | `Literal["packed","shipped"]` | R9 |
| 5 | Few parameters, **no surprise keys** | `additionalProperties: false` | R10, R11 |
| 6 | Writes are **idempotent** and offer **dry-run** | `idempotency_key`, `dry_run` | R12 |
| 7 | Errors **teach**: say what was wrong, what is valid, what to try next | see §4 | (tested by behaviour) |
| 8 | Reads **paginate and summarise**; never dump | `limit`, `cursor`, `detail` | (tested by behaviour) |
| 9 | Fewer, sharper tools beat many overlapping ones | consolidate | `find_overlaps` |

**What the model actually sees.** Not your Python. Just this JSON (printed by `ToolBox.schemas()`): so the docstring *is* the prompt.

In [ ]:
import json, re, random, difflib, copy, itertools, collections
from dataclasses import dataclass
from typing import Literal
from agentkit import tool, ToolBox, get_llm, say, call, run_agent, is_live

@tool
def search(q: str) -> list:
    """Search orders."""
    return []

print(json.dumps(ToolBox(search).schemas()[0], indent=2))
print("\n-> Name 'search', description 'Search orders.', one param 'q' with NO description. Search what? Which orders? How many results? The model must guess.")

## 2 · Build it: a tool linter
Style guides rot; **linters don't**. We encode the principles as twelve rules over the JSON schema (`name`, `description`, `input_schema`), each worth points that sum to 100. A rule returns `(points, max_points, message)` so a failing score tells the author exactly what to fix.

| Rule | Checks | Pts |
|---|---|---|
| R1 | snake_case name that starts with a known verb and has ≥ 2 words | 10 |
| R2 | name is not a generic word (`run`, `data`, `handler`…) | 5 |
| R3 | description length 60-700 chars (25-59 earns half) | 10 |
| R4 | says *when to use* it | 10 |
| R5 | states *boundaries* (do not / only / instead) | 10 |
| R6 | says what it **returns** | 5 |
| R7 | every parameter has a description (≥ 12 chars) | 15 |
| R8 | no ambiguous parameter names (`data`, `id`, `x`…) | 10 |
| R9 | closed-set string params are enums | 10 |
| R10 | ≤ 6 parameters | 5 |
| R11 | `additionalProperties` is false | 5 |
| R12 | *mutating* tools mention idempotency / dry-run / irreversibility / approval | 5 |

In [ ]:
VERBS = {"get", "list", "search", "find", "lookup", "calculate", "estimate", "check", "validate", "create", "update", "cancel", "issue",
         "send", "delete", "remove", "set", "submit", "approve", "charge", "refund", "fetch", "read", "write", "count", "summarize"}
MUTATING = {"create", "update", "cancel", "issue", "send", "delete", "remove", "set", "submit", "approve", "charge", "refund", "write"}
GENERIC = {"run", "do", "execute", "tool", "helper", "process", "handle", "handler", "data", "info", "stuff", "misc", "util", "manage", "main", "action"}
AMBIGUOUS = {"data", "input", "x", "y", "z", "params", "args", "kwargs", "value", "thing", "obj", "info", "q", "id", "key", "opts", "payload"}
CLOSED_NAMES = {"status", "mode", "type", "kind", "currency", "format", "sort", "order_by", "priority", "service", "state", "detail", "level"}

def lint_tool(schema: dict) -> dict:
    """Score one tool schema. Returns {rule: (points, max, message)}."""
    name, desc = schema["name"], schema.get("description", "")
    inp = schema.get("input_schema", {}); props = inp.get("properties", {})
    parts = name.split("_")
    out = {}
    out["R1"] = ((10, 10, "ok") if re.fullmatch(r"[a-z][a-z0-9]*(_[a-z0-9]+)+", name) and parts[0] in VERBS
                 else (0, 10, f"name '{name}' should be snake_case verb_object (e.g. get_order_status)"))
    out["R2"] = ((5, 5, "ok") if not (set(parts) & GENERIC) else (0, 5, f"name '{name}' contains a generic word {sorted(set(parts) & GENERIC)}"))
    n = len(desc)
    out["R3"] = (10, 10, "ok") if 60 <= n <= 700 else ((5, 10, f"description is {n} chars; aim for 60-700") if 25 <= n < 60 else (0, 10, f"description is {n} chars; aim for 60-700"))
    out["R4"] = ((10, 10, "ok") if re.search(r"\buse (this )?(when|if)\b|\bcall this\b|\bwhen (the |a |an )?(user|customer|you|staff)\b", desc, re.I)
                 else (0, 10, "say WHEN to use the tool (e.g. 'Use when the customer asks...')"))
    out["R5"] = ((10, 10, "ok") if re.search(r"\b(do not|don't|not for|never|instead|only)\b", desc, re.I)
                 else (0, 10, "say what it is NOT for / which tool to use instead"))
    out["R6"] = (5, 5, "ok") if re.search(r"\breturns?\b", desc, re.I) else (0, 5, "say what the tool returns")
    short = [p for p, s in props.items() if len(s.get("description", "")) < 12]
    out["R7"] = (round(15 * (1 - len(short) / len(props))), 15, "ok" if not short else f"undocumented params: {short}") if props else (15, 15, "ok")
    amb = [p for p in props if p in AMBIGUOUS]
    out["R8"] = (round(10 * (1 - len(amb) / len(props))), 10, "ok" if not amb else f"ambiguous param names: {amb}") if props else (10, 10, "ok")
    need = [p for p, s in props.items() if s.get("type") == "string" and "enum" not in s and
            (p in CLOSED_NAMES or re.search(r"\b(one of|either|must be|can be|valid values?|options?)\b", s.get("description", ""), re.I))]
    out["R9"] = (10, 10, "ok") if not need else (0, 10, f"closed-set params should be enums: {need}")
    out["R10"] = (5, 5, "ok") if len(props) <= 6 else (0, 5, f"{len(props)} params; split the tool")
    out["R11"] = (5, 5, "ok") if inp.get("additionalProperties") is False else (0, 5, "set additionalProperties: false")
    blob = desc + " " + " ".join([p + " " + s.get("description", "") for p, s in props.items()])
    out["R12"] = ((5, 5, "ok") if parts[0] not in MUTATING or re.search(r"idempot|dry[_ ]run|irreversible|cannot be undone|approval|confirm", blob, re.I)
                  else (0, 5, "mutating tool: state idempotency / add dry_run / flag irreversibility"))
    return out

def score(schema: dict) -> int:
    return sum(p for p, _, _ in lint_tool(schema).values())

def report(schema: dict, quiet=False) -> int:
    res = lint_tool(schema); s = sum(p for p, _, _ in res.values())
    if not quiet:
        print(f"{schema['name']:<26} {s:>3}/100")
        for r, (p, m, msg) in res.items():
            if p < m: print(f"    {r:<4}{p:>2}/{m:<2} {msg}")
    return s

assert sum(m for _, m, _ in lint_tool(ToolBox(search).schemas()[0]).values()) == 100
report(ToolBox(search).schemas()[0])

### Unit-test the linter
A linter nobody tested is a rumour. Take a schema that should score 100, then apply **one mutation at a time** and assert that *exactly the intended rule* loses points. First we need a gold-standard tool, so we write the first real tool of the repair set: the quote calculator.

In [ ]:
@tool
def calculate_shipping_quote(destination_postcode: str, weight_kg: float, service: Literal["standard", "express", "overnight"]) -> dict:
    """Estimate the courier price in NZD to ship a parcel within New Zealand. Use when a customer asks what delivery will cost before ordering. Do not use for orders that already exist (use get_order_status). Returns price_nzd, service and estimated_days as JSON.

    Args:
        destination_postcode: Four-digit NZ postcode, e.g. "9016".
        weight_kg: Parcel weight in kilograms, between 0.1 and 30.
        service: Delivery speed to quote: standard, express or overnight.
    """
    if not re.fullmatch(r"\d{4}", destination_postcode):
        raise ValueError(f"destination_postcode '{destination_postcode}' is not a 4-digit NZ postcode. Example of a valid value: \"9016\". Ask the customer for their postcode.")
    if service not in ("standard", "express", "overnight"):
        raise ValueError(f"service '{service}' is not valid. Choose one of: standard, express, overnight.")
    if not 0.1 <= weight_kg <= 30:
        raise ValueError(f"weight_kg {weight_kg} is outside 0.1-30. For parcels over 30 kg tell the customer we need a freight quote from a human.")
    base = {"standard": 6.5, "express": 12.0, "overnight": 22.0}[service]
    return {"price_nzd": round(base + 1.8 * weight_kg, 2), "service": service, "estimated_days": {"standard": 3, "express": 2, "overnight": 1}[service]}

GOLD = ToolBox(calculate_shipping_quote).schemas()[0]
assert score(GOLD) == 100, lint_tool(GOLD)

def mutate(fn):
    s = copy.deepcopy(GOLD); fn(s); return lint_tool(s)

def lost(res):   # which rules lost points
    return {r for r, (p, m, _) in res.items() if p < m}

assert lost(mutate(lambda s: s.update(name="run"))) == {"R1", "R2"}
assert lost(mutate(lambda s: s.update(name="shipping_quote"))) == {"R1"}
assert lost(mutate(lambda s: s.update(description="Gets a quote."))) == {"R3", "R4", "R5", "R6"}
assert lost(mutate(lambda s: s.update(description=s["description"].replace("Use when", "Handy if")))) == {"R4"}
assert lost(mutate(lambda s: s["input_schema"]["properties"]["weight_kg"].pop("description"))) == {"R7"}
assert lost(mutate(lambda s: s["input_schema"]["properties"].update(data={"type": "string", "description": "Some extra data to include here."}))) == {"R8"}
assert lost(mutate(lambda s: s["input_schema"]["properties"]["service"].pop("enum"))) == {"R9"}
assert lost(mutate(lambda s: s["input_schema"]["properties"].update({f"p{i}": {"type": "number", "description": "Extra numeric option here."} for i in range(5)}))) == {"R10"}
assert lost(mutate(lambda s: s["input_schema"].update(additionalProperties=True))) == {"R11"}
assert lost(mutate(lambda s: s.update(name="cancel_shipment"))) == {"R12"}
print("linter: 10 mutation tests pass; each rule fires on exactly the defect it targets")

## 3 · Use it: five bad tools, five fixes
Here is Kauri Outdoors' toolbox as it was written. First the shared synthetic "database" (seeded; 120 orders with deliberately bulky records), then the five bad tools.

In [ ]:
rng = random.Random(7)
FIRST = ["aroha", "wiremu", "mere", "tane", "hine", "anika", "liam", "sofia"]
ORDERS = {}
for i in range(1000, 1120):
    st = rng.choice(["packed", "shipped", "delivered", "cancelled"])
    ORDERS[str(i)] = {
        "order_id": str(i), "customer_email": f"{rng.choice(FIRST)}@example.co.nz", "status": st, "total": round(rng.uniform(25, 420), 2),
        "delivered_days_ago": rng.randint(0, 60) if st == "delivered" else None,
        "items": [{"sku": f"SKU-{rng.randint(100, 999)}", "qty": rng.randint(1, 3), "description": "Merino base layer, long sleeve, 190gsm, responsibly sourced wool. " * 2} for _ in range(rng.randint(1, 4))],
        "address": {"line1": f"{rng.randint(1, 200)} Kauri Road", "city": rng.choice(["Dunedin", "Wellington", "Auckland"]), "postcode": str(rng.randint(1000, 9999))},
        "history": [{"ts": f"2026-0{rng.randint(1, 9)}-1{rng.randint(0, 9)}T09:00", "event": "status_change", "note": "automated update from warehouse system"} for _ in range(5)],
    }
BAD_LEDGER = []

@tool
def run(data: dict, mode: str) -> dict:
    """Runs stuff."""
    return {}

@tool
def get_info(id: str) -> dict:
    """Gets info."""
    return ORDERS[id]            # KeyError on anything unexpected

@tool
def update(order: str, status: str, notify: bool) -> str:
    """Update an order."""
    ORDERS[order]["status"] = status
    return "ok"

@tool
def bad_search(q: str) -> list:
    """Search orders."""
    return [o for o in ORDERS.values() if q.lower() in json.dumps(o).lower()]

@tool
def refund_order(order_id: str, amount: float) -> str:
    """Refund an order."""
    BAD_LEDGER.append((order_id, amount))
    return "done"

BAD = ToolBox(run, get_info, update, bad_search, refund_order)
bad_scores = {s["name"]: report(s) for s in BAD.schemas()}
mean_bad = sum(bad_scores.values()) / len(bad_scores)
print(f"\nmean score of the original five: {mean_bad:.0f}/100")
assert mean_bad < 45

### The repairs
Each fix maps to a principle, not just to a lint score:

| Bad tool | Problem | Repair |
|---|---|---|
| `run(data, mode)` | god-tool; opaque | split by *job*: `calculate_shipping_quote` (written above) |
| `get_info(id)` | which info? which id? | `get_order_status(order_id)`; compact result; **guided errors** |
| `update(order, status, notify)` | free-text `status` accepts "shiped" | `update_order_status` with a `Literal` enum, allowed transitions, idempotent |
| `search(q)` | returns everything, unbounded | `search_orders` with filters, `limit`, `cursor`, `detail` |
| `refund_order(...)` | retry = double refund | `issue_refund` with `idempotency_key` + `dry_run` + policy checks |

In [ ]:
def _guide_order_id(order_id: str) -> str:
    digits = re.sub(r"\D", "", order_id)
    msg = f"No order with id '{order_id}'. Order IDs are 4-digit numbers such as \"1042\"."
    if digits in ORDERS and digits != order_id:
        msg += f" Did you mean \"{digits}\"?"
    else:
        close = difflib.get_close_matches(digits, list(ORDERS), n=2, cutoff=0.75)
        if close: msg += f" Closest known IDs: {close}."
    return msg + " If you only know the customer, call search_orders(customer_email=...) to find the ID."

@tool
def get_order_status(order_id: str) -> dict:
    """Look up the current status and total of ONE order by its order ID. Use when the customer asks where their order is or whether it has shipped. Do not use to find orders by email (use search_orders). Returns order_id, status and total as JSON.

    Args:
        order_id: Four-digit order ID such as "1042" (digits only, no # prefix).
    """
    o = ORDERS.get(order_id)
    if o is None: raise KeyError(_guide_order_id(order_id))
    return {"order_id": o["order_id"], "status": o["status"], "total": o["total"]}

_ALLOWED = {("packed", "shipped"), ("packed", "cancelled")}

@tool
def update_order_status(order_id: str, new_status: Literal["packed", "shipped", "cancelled"], notify_customer: bool = True) -> str:
    """Move an order to a new status: packed to shipped, or packed to cancelled. Use when warehouse or support staff confirm the change. Do not use for refunds (use issue_refund). Idempotent: repeating the same change is safe and reports no change. Returns the new status, or an error that explains which transitions are allowed.

    Args:
        order_id: Four-digit order ID such as "1042".
        new_status: Target status; shipped orders cannot be cancelled.
        notify_customer: Send the customer an email about the change. Default true.
    """
    o = ORDERS.get(order_id)
    if o is None: raise KeyError(_guide_order_id(order_id))
    if new_status not in ("packed", "shipped", "cancelled"):
        raise ValueError(f"new_status '{new_status}' is not valid. Choose one of: packed, shipped, cancelled.")
    if o["status"] == new_status:
        return f"Order {order_id} is already '{new_status}'; no change made."
    if (o["status"], new_status) not in _ALLOWED:
        raise ValueError(f"Cannot move order {order_id} from '{o['status']}' to '{new_status}'. Allowed: packed->shipped, packed->cancelled. "
                         "For a delivered order use issue_refund instead.")
    o["status"] = new_status
    return f"Order {order_id} is now '{new_status}'." + (" Customer notified." if notify_customer else "")

@tool
def search_orders(customer_email: str = "", status: Literal["any", "packed", "shipped", "delivered", "cancelled"] = "any",
                  limit: int = 10, cursor: str = "", detail: Literal["concise", "full"] = "concise") -> dict:
    """Find orders by customer email and/or status, in order-ID order. Use when you do not have an order ID yet. Do not use to read one known order (use get_order_status). Returns one page of results with total, next_cursor and a hint; pass next_cursor back to get the next page.

    Args:
        customer_email: Exact customer email, e.g. "aroha@example.co.nz". Empty matches any customer.
        status: Filter by order status; any means no filter.
        limit: Page size from 1 to 50. Default 10.
        cursor: The next_cursor from a previous page; empty for the first page.
        detail: concise gives id, status and total; full adds items, address and history.
    """
    if not 1 <= limit <= 50:
        raise ValueError(f"limit {limit} is out of range. Use 1-50; to read more, page with next_cursor.")
    hits = [o for o in ORDERS.values() if (not customer_email or o["customer_email"] == customer_email) and (status == "any" or o["status"] == status)]
    start = int(cursor or 0)
    page = hits[start:start + limit]
    nxt = str(start + limit) if start + limit < len(hits) else None
    items = [{k: o[k] for k in ("order_id", "status", "total")} if detail == "concise" else o for o in page]
    hint = ("No matches: check the email spelling or try status='any'." if not hits else
            f"Showing {start + 1}-{start + len(page)} of {len(hits)}." + (" Call again with next_cursor for more, or narrow the filters." if nxt else ""))
    return {"total": len(hits), "items": items, "next_cursor": nxt, "hint": hint}

REFUNDS = {}   # idempotency_key -> record (our "payments system")

@tool
def issue_refund(order_id: str, amount: float, idempotency_key: str, dry_run: bool = False) -> str:
    """Refund a DELIVERED order, in full or part, to the original payment method. Use when policy allows (within 30 days of delivery) and the customer asked for money back. Do not use for undelivered orders; cancel them instead. Idempotent per idempotency_key: retrying with the same key never refunds twice. Set dry_run to true to preview without moving money. Returns a confirmation, or an error explaining what to fix.

    Args:
        order_id: Four-digit order ID such as "1042".
        amount: Amount in NZD, greater than 0 and at most the remaining refundable total.
        idempotency_key: Unique key for this refund intent, e.g. "refund-1042-001". Reuse the same key when retrying.
        dry_run: If true, validate and preview only; nothing is changed.
    """
    if idempotency_key in REFUNDS:
        r = REFUNDS[idempotency_key]
        return f"Already processed: refund ${r['amount']:.2f} for order {r['order_id']} (key {idempotency_key}). No new refund was made."
    o = ORDERS.get(order_id)
    if o is None: raise KeyError(_guide_order_id(order_id))
    if o["status"] != "delivered":
        raise ValueError(f"Order {order_id} is '{o['status']}', not delivered, so it cannot be refunded. If it has not shipped, use update_order_status(new_status='cancelled').")
    if o["delivered_days_ago"] > 30:
        raise ValueError(f"Order {order_id} was delivered {o['delivered_days_ago']} days ago; policy allows 30. Escalate to a human if the customer disputes this.")
    remaining = round(o["total"] - sum(r["amount"] for r in REFUNDS.values() if r["order_id"] == order_id), 2)
    if not 0 < amount <= remaining:
        raise ValueError(f"amount {amount} is invalid; refundable remaining for order {order_id} is ${remaining:.2f}.")
    if dry_run:
        return f"DRY RUN: would refund ${amount:.2f} on order {order_id} (remaining after: ${remaining - amount:.2f}). Nothing was changed."
    REFUNDS[idempotency_key] = {"order_id": order_id, "amount": amount}
    return f"Refunded ${amount:.2f} on order {order_id} (key {idempotency_key})."

GOOD = ToolBox(calculate_shipping_quote, get_order_status, update_order_status, search_orders, issue_refund)
good_scores = {s["name"]: report(s, quiet=True) for s in GOOD.schemas()}
print({k: v for k, v in good_scores.items()})
mean_good = sum(good_scores.values()) / len(good_scores)
print(f"\nmean score: original {mean_bad:.0f}  ->  repaired {mean_good:.0f}")
assert min(good_scores.values()) >= 95 and mean_bad < 45
print("the model now sees:\n", json.dumps(GOOD.schemas()[2]["input_schema"]["properties"]["new_status"]))

## 4 · Behaviour matters as much as schema
A linter reads *text*. Principles 6-8 are about what the tool **does** when things go wrong or get big. Each demo below runs real code and asserts the property.

### 4a · Errors that teach recovery
The model sends `order_id="#A1042"` (copying the customer's phrasing). Compare what comes back to the model.

In [ ]:
bad_msg, _ = BAD.run("get_info", {"id": "#A1042"})
good_msg, is_err = GOOD.run("get_order_status", {"order_id": "#A1042"})
print("BAD :", bad_msg); print("GOOD:", good_msg)
assert is_err and "KeyError" in bad_msg and "Did you mean" not in bad_msg
assert '"1042"' in good_msg and "search_orders" in good_msg      # says what is valid AND what to do next

Does a better message actually change the outcome? Replay the agent loop: the model's first call has the malformed ID, reads the guided error, and *fixes it on the second call*. (In LIVE mode Claude does this on its own; the guided message is why it can.)

In [ ]:
SYSTEM = "You are Kauri Outdoors' support agent. Use tools for facts. If a tool errors, read the message and fix your call."
llm = get_llm(script=[call("get_order_status", order_id="#A1042"),
                      call("get_order_status", order_id="1042", _say="The ID should be digits only; retrying."),
                      say(f"Order 1042 is currently '{ORDERS['1042']['status']}'.")])
res = run_agent(llm, GOOD, "Where is order #A1042?", system=SYSTEM, max_turns=5)
res.tracer.show()
assert res.tracer.n_tool_calls() == 2 and res.stop == "end_turn"
errs = [e for e in res.tracer.events if e.kind == "tool"]
assert errs[0].detail["error"] and not errs[1].detail["error"]

### 4b · Idempotency and dry-run: the retry that double-charges
The classic failure: the refund *succeeds*, the HTTP response is lost, the agent (reasonably!) retries. Without an idempotency key the retry is a second refund.

In [ ]:
oid = next(k for k, o in ORDERS.items() if o["status"] == "delivered" and o["delivered_days_ago"] <= 30)
amt = ORDERS[oid]["total"]
for _ in range(2):                                   # "timeout -> retry" against the ORIGINAL tool
    BAD.run("refund_order", {"order_id": oid, "amount": amt})
print("original tool ledger :", BAD_LEDGER, "-> customer refunded TWICE")
assert len(BAD_LEDGER) == 2

REFUNDS.clear()
preview, _ = GOOD.run("issue_refund", {"order_id": oid, "amount": amt, "idempotency_key": "refund-001", "dry_run": True})
assert REFUNDS == {} and "DRY RUN" in preview                       # preview moves no money
for _ in range(2):                                   # same retry, same key
    out, err = GOOD.run("issue_refund", {"order_id": oid, "amount": amt, "idempotency_key": "refund-001"})
print("repaired tool        :", out)
assert len(REFUNDS) == 1 and "Already processed" in out and not err
over, err = GOOD.run("issue_refund", {"order_id": oid, "amount": 1.0, "idempotency_key": "refund-002"})
print("over-refund guard    :", over); assert err                      # remaining total is exhausted

*Design note.* `dry_run` also gives you a cheap **approval pattern** (AG09): the agent previews, a human sees the exact consequence, and approval triggers the real call with the same key.

### 4c · Pagination and truncation: don't flood the context window
`ToolBox.run` truncates any result at `max_result_chars` (8000 here) **mid-string**, which silently drops data and may cut JSON in half. The original `search` returns whole records for every substring match.

In [ ]:
bad_out, _ = BAD.run("bad_search", {"q": "shipped"})
n_match = sum(1 for o in ORDERS.values() if "shipped" in json.dumps(o).lower())
n_seen = bad_out.count('"order_id"')
print(f"original search: {n_match} matches, {len(bad_out):,} chars returned, only {n_seen} records visible before the cut")
print("tail:", repr(bad_out[-60:]))
assert "[truncated" in bad_out and n_seen < n_match           # the model silently loses records

pages, cur = [], ""
while True:
    out, err = GOOD.run("search_orders", {"status": "shipped", "limit": 7, "cursor": cur})
    assert not err; d = json.loads(out); pages.append(d)
    if not d["next_cursor"]: break
    cur = d["next_cursor"]
ids = [i["order_id"] for p in pages for i in p["items"]]
expected = [o["order_id"] for o in ORDERS.values() if o["status"] == "shipped"]
print(f"repaired search: {len(pages)} pages, first page {len(json.dumps(pages[0])):,} chars, hint = {pages[0]['hint']!r}")
assert ids == expected and len(set(ids)) == len(ids)          # complete, ordered, no duplicates
assert len(json.dumps(pages[0])) < 1500
too_big, err = GOOD.run("search_orders", {"limit": 500}); print("limit=500 ->", too_big); assert err and "next_cursor" in too_big

## 5 · More tools ≠ better agent: measuring tool-selection accuracy
The spec says *"tool-count vs accuracy"*, so measure it. The experiment: 15 natural customer-support requests, each with one correct tool. For each toolbox configuration we show the model the request plus the tool list (forcing a tool call with `tool_choice={"type": "any"}`) and check which tool it picked.

* **Config A:** the five *original* bad tools. **Config B:** the five repaired tools. **Config C/D:** the repaired five **plus 10 / 30 plausible neighbouring tools** (`get_order_history`, `list_customer_orders`, `cancel_order`, …) whose descriptions overlap.

**Offline caveat (important).** Without a key the "model" is a `PolicyLLM`: a transparent **lexical matcher** (IDF-weighted word overlap between the request and each tool's name, description and parameters). It is *not* Claude and its absolute numbers mean nothing. It is a controlled stand-in that can only use what is written in the schema, so it isolates the effect of the *text you wrote*. Run LIVE (≈ 60 short calls) to get the real model's numbers; the harness is identical.

In [ ]:
LABELLED = [  # (customer request, correct repaired tool)
    ("How much would shipping a 3 kg parcel to postcode 9016 cost with express?", "calculate_shipping_quote"),
    ("Estimate the courier price for 12 kg to 6011 overnight", "calculate_shipping_quote"),
    ("What does delivery cost for a 1.5 kg parcel to 1010?", "calculate_shipping_quote"),
    ("Where is order 1042 right now?", "get_order_status"),
    ("Has order 1050 shipped yet?", "get_order_status"),
    ("What is the current status of order 1077?", "get_order_status"),
    ("Mark order 1042 as shipped", "update_order_status"),
    ("Cancel order 1033 before it goes out", "update_order_status"),
    ("Move order 1090 to packed and email the customer", "update_order_status"),
    ("Find all orders from aroha@example.co.nz", "search_orders"),
    ("Show shipped orders for wiremu@example.co.nz", "search_orders"),
    ("List the cancelled orders belonging to mere@example.co.nz", "search_orders"),
    ("Refund $129 on order 1042", "issue_refund"),
    ("Give the customer their money back for order 1055", "issue_refund"),
    ("Issue a refund of 40 dollars for order 1061, the item arrived damaged", "issue_refund"),
]
BAD_OF = {"calculate_shipping_quote": "run", "get_order_status": "get_info", "update_order_status": "update",
          "search_orders": "bad_search", "issue_refund": "refund_order"}

DISTRACTORS = [
    ("get_order_invoice", "Download the PDF tax invoice for an order."), ("get_order_history", "Return the timeline of status events for an order."),
    ("list_customer_orders", "List every order placed by a customer email address."), ("cancel_order", "Cancel an order that has not shipped."),
    ("update_shipping_address", "Change the delivery address of an order that has not shipped."), ("track_parcel", "Get courier tracking events for a shipped parcel."),
    ("get_refund_status", "Check whether a refund for an order has been processed."), ("list_refunds", "List refunds issued in a date range."),
    ("estimate_delivery_date", "Estimate the delivery date for an order from its shipping service."), ("get_shipping_rates", "List all courier services and prices available for a destination postcode."),
    ("validate_postcode", "Check that a postcode exists in New Zealand."), ("get_product_stock", "Return units in stock for a product SKU."),
    ("search_products", "Search the product catalogue by keyword."), ("get_customer_profile", "Return the profile for a customer email address."),
    ("update_customer_email", "Change the email address stored for a customer."), ("send_order_confirmation_email", "Resend the order confirmation email to the customer."),
    ("create_return_label", "Create a prepaid return label for a delivered order."), ("get_return_status", "Check where a customer return is in the process."),
    ("apply_discount_code", "Apply a promotion code to an open order."), ("get_payment_status", "Return payment status for an order: paid, pending or failed."),
    ("capture_payment", "Capture an authorised card payment for an order."), ("list_warehouses", "List the warehouses that stock and ship orders."),
    ("get_warehouse_stock", "Return units on hand for a SKU in one warehouse."), ("create_support_ticket", "Open a support ticket for a customer."),
    ("update_support_ticket", "Add a note or change the status of a support ticket."), ("search_support_tickets", "Search support tickets by customer or status."),
    ("get_order_total", "Return the total amount of an order."), ("get_gift_card_balance", "Return the remaining balance of a gift card."),
    ("redeem_gift_card", "Redeem a gift card against an order."), ("list_shipping_services", "List the shipping services we offer: standard, express, overnight."),
]
def distractor_schemas(n):
    return [{"name": nm, "description": d, "input_schema": {"type": "object", "properties": {}, "additionalProperties": False}} for nm, d in DISTRACTORS[:n]]

STOP = set("a an the of to for and or is it in on at my me i we you with that this be as by from do does can please what how where when show give tell all their they them his her its our".split())
def _toks(s):
    out = []
    for w in re.findall(r"[a-z]+", s.lower()):
        if w in STOP or len(w) < 3: continue
        for suf in ("ing", "ed", "es", "s"):
            if w.endswith(suf) and len(w) - len(suf) >= 4:
                w = w[: -len(suf)]
                if len(w) > 4 and w[-1] == w[-2]: w = w[:-1]      # shipp -> ship, cancell -> cancel
                break
        out.append(w)
    return out

def lexical_policy(messages, system, tools):
    """Offline stand-in for a model: pick the tool whose text overlaps the request most (IDF-weighted). Uses ONLY the schema text."""
    q = set(_toks(messages[-1]["content"]))
    docs = []
    for t in tools:
        props = t.get("input_schema", {}).get("properties", {})
        desc = re.sub(r"\b(do not|don't)[^.]*\.", "", t.get("description", ""), flags=re.I)       # a reader understands "Do not use for X"
        text = " ".join([t["name"].replace("_", " ")] * 2 + [desc] + [p + " " + s.get("description", "") for p, s in props.items()])
        docs.append(_toks(text))
    df = collections.Counter(w for d in docs for w in set(d))
    idf = {w: 1 + (len(docs) / df[w]) for w in df}
    scores = [sum(idf[w] for w in q & set(d)) / (1 + 0.005 * len(d)) for d in docs]
    best = max(range(len(tools)), key=lambda i: scores[i])       # ties -> first tool in the list
    return call(tools[best]["name"])

def selection_accuracy(llm, schemas, gold_name):
    ok = 0
    for q, gold in LABELLED:
        r = llm.chat([{"role": "user", "content": q}], system="Pick the single best tool for the request and call it. Always respond with exactly one tool call.",
                     tools=schemas, max_tokens=200)   # no forced tool_choice: current models reject it; steer via the prompt
        
        ok += bool(r.tool_calls) and r.tool_calls[0]["name"] == gold_name(gold)
    return ok / len(LABELLED)

llm = get_llm(policy=lexical_policy)
configs = {"A: original 5": (BAD.schemas(), lambda g: BAD_OF[g]),
           "B: repaired 5": (GOOD.schemas(), lambda g: g),
           "C: repaired 5 + 10": (GOOD.schemas() + distractor_schemas(10), lambda g: g),
           "D: repaired 5 + 30": (GOOD.schemas() + distractor_schemas(30), lambda g: g)}
acc = {k: selection_accuracy(llm, s, gn) for k, (s, gn) in configs.items()}
for k, v in acc.items(): print(f"{k:<22} selection accuracy = {v:.0%}  ({round(v * len(LABELLED))}/{len(LABELLED)})")

In [ ]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(range(len(acc)), list(acc.values()), color=["#e45756", "#4c78a8", "#f58518", "#f58518"])
ax.set_xticks(range(len(acc))); ax.set_xticklabels([k.replace(": ", "\n") for k in acc], fontsize=8)
ax.set_ylim(0, 1.05); ax.set_ylabel("selection accuracy"); ax.set_title(("LIVE model" if is_live() else "OFFLINE lexical proxy") + ": 15 requests")
plt.show()
if not is_live():
    assert acc["B: repaired 5"] >= 0.75, acc
    assert acc["A: original 5"] <= acc["B: repaired 5"] - 0.2, acc          # description quality is worth a lot
    assert acc["D: repaired 5 + 30"] <= acc["B: repaired 5"] - 0.1, acc     # clutter costs accuracy; C and D both lose ground
print("Read it as: (1) text quality is the lever the proxy can see; (2) every extra overlapping tool is another way to be wrong.")

**What this does and does not show.** Frontier models tolerate much more clutter than this proxy; what does *not* go away is the structure of the problem: each additional tool adds more definitions to the context (cost on every turn, see AG01 §5b) and more near-neighbours to confuse. Practical consequences: **consolidate** tools that always run together, **namespace** by service (`crm_`, `billing_`), and, where your platform offers it, load rarely-used tool definitions on demand (check current docs for your provider's mechanism). Measure on *your* requests before believing anyone's rule of thumb, including this notebook's.

## 6 · Break it
### 6a · Portfolio linting: near-duplicate tools
Per-tool lint cannot see that two tools are the *same* tool under different names, which is a leading cause of flip-flopping tool choice. Add a toolbox-level check: Jaccard similarity of description word-sets.

In [ ]:
def find_overlaps(schemas, thresh=0.5):
    sets = {s["name"]: set(_toks(s["description"])) for s in schemas}
    out = []
    for a, b in itertools.combinations(sets, 2):
        j = len(sets[a] & sets[b]) / max(1, len(sets[a] | sets[b]))
        if j >= thresh: out.append((a, b, round(j, 2)))
    return out

@tool
def lookup_order(order_id: str) -> dict:
    """Look up an order by ID and return its status and total."""
    return {}

@tool
def get_order_details(order_id: str) -> dict:
    """Get an order by ID and return its status and total."""
    return {}

dupes = find_overlaps(ToolBox(lookup_order, get_order_details).schemas())
print("near-duplicates:", dupes)
assert dupes and dupes[0][2] >= 0.5
assert find_overlaps(GOOD.schemas()) == [], "the repaired five are distinct"

### 6b · A model that ignores the enum
Enums in the schema are *guidance*, not a firewall: a model can still emit `"shiped"`, and a client/SDK may not validate. **Always re-validate in code** and make the error say what is valid. Verify our tool does:

In [ ]:
msg, err = GOOD.run("update_order_status", {"order_id": "1042", "new_status": "shiped"})
print(msg); assert err and "packed, shipped, cancelled" in msg
msg2, err2 = GOOD.run("update_order_status", {"order_id": "1042", "new_status": "shipped", "bogus_key": 1})
print(msg2); assert err2 and "Bad arguments" in msg2           # surprise keys are rejected, not silently ignored
packed = next(k for k, o in ORDERS.items() if o["status"] == "packed")
a, _ = GOOD.run("update_order_status", {"order_id": packed, "new_status": "shipped"})
b, _ = GOOD.run("update_order_status", {"order_id": packed, "new_status": "shipped"})
print(a, "|", b); assert "already" in b                         # idempotent

## 7 · Exercises
**Practice (30 min)**
1. Write a *seventh* repaired tool, `get_order_history(order_id, limit=5)`, that returns the last `limit` history events (concise), with a guided error for unknown IDs. Make `score(schema) == 100` and `find_overlaps` stay empty against `GOOD`.
2. Add an **R13** to the linter: penalise descriptions that start with the tool's own name or with "This tool". Add a mutation test for it.

**Stretch (60 min)**
3. Extend `search_orders` with a `fields: list[str]` parameter and validate it against an allow-list with a guided error. Does the linter's R9 see a problem? (It is a *list* of closed values; extend R9.)
4. Re-run the selection experiment with 15 *new* requests you write yourself, including 3 deliberately ambiguous ones. Which tool pair do they confuse, and how would you fix the descriptions?

**Challenge (2 h, a key helps)**
5. Run §5 LIVE with `AGENTKIT_MODEL` set to a small and a large model. Plot accuracy vs number of distractors (0, 5, 10, 20, 30) for both, with the *bad* descriptions as a baseline. Report whether description quality or model size moves the curve more, with the evidence.

In [ ]:
# Starter for exercise 1
@tool
def get_order_history(order_id: str, limit: int = 5) -> dict:
    """TODO: write a description that passes R3-R6 (when to use, when NOT to, what it returns).

    Args:
        order_id: TODO
        limit: TODO
    """
    raise NotImplementedError

# s = ToolBox(get_order_history).schemas()[0]
# assert score(s) == 100, lint_tool(s)
# assert find_overlaps(GOOD.schemas() + [s]) == []
# out, err = ToolBox(get_order_history).run("get_order_history", {"order_id": "nope"}); assert err and "4-digit" in out

## ✅ Checkpoint (answer before moving on)
1. A model keeps calling `search_orders` when it should call `get_order_status`. Name two schema changes and one *behavioural* change that reduce this.
2. Why is `additionalProperties: false` worth having if the model "usually" sends the right keys?
3. The refund tool times out client-side after the money already moved. What property makes the agent's retry safe, and where does the key come from?
4. Your offline selection experiment shows 95 % accuracy. Give two reasons not to trust that as a claim about Claude.

<details><summary>Answers</summary>

1. Schema: add "Do not use to read one known order (use get_order_status)" to `search_orders`, and "Use when you have an order ID" to the other; rename or consolidate if they still overlap. Behaviour: have `search_orders` return a hint ("if you already have an ID use get_order_status") and make the *wrong* use cheap, with concise, paginated results.
2. It turns a silent misuse (an ignored or misspelled key) into an immediate, explainable error the model can fix, and keeps the contract closed so the tool never has to guess what an extra key meant.
3. **Idempotency** per `idempotency_key`: the same key returns the original result without a second refund. The *agent* (or the harness) generates a unique key per refund *intent*, and reuses it on retries.
4. The "model" was a lexical matcher that reads only schema text, so it measures description overlap, not reasoning; and the 15 requests were written by the same author as the descriptions (leakage / tiny sample). A live run on held-out, real tickets is needed.
</details>

## 📌 Key takeaways
- The model sees **name + description + schema**, nothing else. Treat them as a prompt, and lint them like code.
- Name `verb_object`; say **when**, **when not**, **what returns**; document every param; use **enums**; close the object.
- **Errors are instructions**: state what was wrong, what is valid, and what to try next.
- **Writes**: idempotency keys and dry-run. **Reads**: limit, cursor, concise/full, and truncation you control.
- Selection accuracy falls with vague text and overlapping neighbours; consolidate, namespace, and *measure on your own requests*.
- Schema is guidance; **re-validate in code**.

## 📚 Further reading
- Anthropic Engineering, *Building effective agents*: the section on prompt-engineering your tools (the agent-computer interface) — anthropic.com/engineering/building-effective-agents
- Anthropic Engineering, *Writing effective tools for agents*
- Anthropic docs: *Tool use with Claude* (tool definitions, `tool_choice`, handling tool results and errors) — docs.anthropic.com
- Yao et al., *ReAct: Synergizing Reasoning and Acting in Language Models* (2022), arXiv:2210.03629

**Next → AG04 · Planning & Reasoning Patterns** (ReAct, plan-and-execute and reflection, with tools you can now trust).